# 高级上下文工程:智能体的大脑

欢迎来到 Semantica 上下文工程的**大师课**。本 Notebook 演示如何为你的 AI 智能体构建生产级的记忆系统。

与会话结束就忘记一切的简单聊天机器人不同,**具备上下文感知的智能体**需要:
*   **长期记忆**:能回忆起几周前的事实。
*   **结构化知识**:理解实体(人物、项目、主题)之间是如何关联的。
*   **混合检索**:把模糊的文本搜索与精确的图遍历结合起来。

## 学习目标

在本次演练中,我们将:
1.  **初始化生产级存储**:用真实的**向量库**(FAISS)和**图存储**(Neo4j)取代玩具示例。
2.  **构建智能体上下文**:配置负责统筹记忆的中央大脑。
3.  **接入知识**:存储复杂文档并自动抽取实体。
4.  **注入关系**:手动教会智能体了解世界中的关联。
5.  **执行 GraphRAG**:运行高级查询,在知识图谱中"跳转"以找到标准 RAG 遗漏的答案。
6.  **管理生命周期**:学习清理旧记忆,保持系统健康。

---

## 1. 安装

开始之前,只需安装该包:

```bash
pip install semantica
```

In [ ]:
!pip install -qU semantica 

In [ ]:
import sys
import os
import time
from typing import Any, List, Dict, Optional

# Add project root to path to import semantica
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../../")))

# Core Imports
from semantica.context import AgentContext, ContextGraph, AgentMemory
from semantica.vector_store import VectorStore
from semantica.graph_store import GraphStore

print("Libraries imported successfully.")

---

## 2. 初始化存储后端

现在我们连接持久化存储层。Semantica 把它们抽象在统一的接口之后,因此你可以更换后端(例如从 FAISS 切换到 Weaviate)而无需修改应用逻辑。

### 向量库(图书馆)
存放记忆和文档的*内容*,并按语义进行索引。

In [ ]:
try:
    # Initialize FAISS Vector Store
    # You can also use: backend="weaviate", backend="qdrant", etc.
    vs = VectorStore(backend="faiss", dimension=768)
    print("VectorStore initialized (Backend: FAISS)")
except ImportError:
    print("FAISS not installed. Using in-memory fallback (not persistent).")
    vs = VectorStore(backend="inmemory", dimension=768)
except Exception as e:
    print(f"VectorStore Error: {e}")
    vs = None

### 图存储(地图)
存放实体之间的*连接*,这对推理至关重要。

In [ ]:
try:
    # Initialize Neo4j Graph Store
    # Ensure your Docker container is running!
    gs = GraphStore(
        backend="neo4j",
        uri="bolt://localhost:7687",
        user="neo4j",
        password="password"
    )
    
    # Test connection
    if gs.connect():
        print("GraphStore connected (Backend: Neo4j)")
    else:
        raise ConnectionError("Could not connect to Neo4j")

except Exception as e:
    print(f"GraphStore Connection Failed: {e}")
    print("   Switching to in-memory ContextGraph (Non-persistent fallback)")
    gs = ContextGraph() # Fallback implementation

---

## 3. 智能体上下文(Agent Context)

`AgentContext` 是高层的编排器。它位于向量库和图存储之上,管理信息的流动。

**GraphRAG 的配置:**
*   `use_graph_expansion=True`:检索时不仅看文档本身,还要看它的邻居。
*   `max_expansion_hops=2`:遍历多远?(例如 A -> B -> C)。
*   `hybrid_alpha=0.6`:权重。0.0 为纯向量,1.0 为纯图。0.6 略偏向图。

In [ ]:
if vs:
    context = AgentContext(
        vector_store=vs,
        knowledge_graph=gs,
        retention_days=90,          # Remember things for 3 months
        use_graph_expansion=True,   # Enable GraphRAG
        max_expansion_hops=2,       # 2-Hop reasoning
        hybrid_alpha=0.6            # Balanced retrieval
    )
    print("Agent Context is online and ready.")
else:
    print("Cannot proceed without VectorStore.")

---

## 4. 接入:教会智能体

我们可以存储不同类型的信息。系统足够聪明,能区分对话记忆和事实性文档。

### A. 情景记忆(对话)
这些是交互的原始日志,提供"个人"历史。

In [ ]:
user_id = "user_123"
session_id = "session_alpha"

# Store a user preference
mem_id = context.store(
    content="I am working on a new project called 'Project Apollo' which uses Python and React.",
    conversation_id=session_id,
    user_id=user_id,
    metadata={"type": "user_preference"}
)
print(f"Memory Stored: {mem_id}")

### B. 语义知识(文档)
当我们输入文档时,希望**抽取实体**并将它们**关联起来**。

*(注意:在真实部署中,这会使用 LLM 来解析实体。这里我们使用上下文模块自带的原生抽取能力。)*

In [ ]:
documents = [
    {
        "content": "Project Apollo is a next-gen web framework designed for high scalability.",
        "metadata": {"source": "internal_wiki", "category": "projects"}
    },
    {
        "content": "Python 3.12 introduces significant performance improvements for async workloads.",
        "metadata": {"source": "tech_news", "category": "languages"}
    }
]

# Store documents and trigger graph build
stats = context.store(
    documents,
    extract_entities=True,      # Extract entities from text
    extract_relationships=True, # Infer relationships
    link_entities=True          # Connect to existing graph nodes
)

print("Knowledge Ingestion Stats:", stats)

---

## 5. 图工程:手动注入

有时自动抽取并不够。你可能想强制执行特定的业务逻辑或关系。我们可以使用 `build_graph` 手动注入节点和边。

**我们将定义:**
*   **用户**(Alice)
*   **角色**(Admin)
*   **项目**(Apollo)
*   **关系**:Alice *MANAGES*(管理)Apollo 项目。

In [ ]:
# 1. Define Nodes
entities = [
    {"id": "alice", "type": "PERSON", "text": "Alice", "properties": {"role": "Admin"}},
    {"id": "project_apollo", "type": "PROJECT", "text": "Project Apollo"},
    {"id": "python", "type": "TECH", "text": "Python"},
    {"id": "react", "type": "TECH", "text": "React"}
]

# 2. Define Edges (The Knowledge)
relationships = [
    {"source": "alice", "target": "project_apollo", "type": "MANAGES", "weight": 1.0},
    {"source": "project_apollo", "target": "python", "type": "USES_TECH", "weight": 1.0},
    {"source": "project_apollo", "target": "react", "type": "USES_TECH", "weight": 1.0}
]

# 3. Inject into Graph
graph_stats = context.build_graph(
    entities=entities,
    relationships=relationships
)

print("Manual Graph Build Complete:", graph_stats)

### 可视化图逻辑
让我们直接查询图,看看 "Project Apollo" 是什么样子。

In [ ]:
# Helper to print graph neighbors
def inspect_node(node_id):
    if hasattr(gs, "get_neighbors"):
        neighbors = gs.get_neighbors(node_id)
        print(f"\nNeighbors of '{node_id}':")
        for n in neighbors:
            # Handle different return formats between stores
            rel_type = n.get('relationship') or n.get('type') or 'linked'
            target = n.get('id') or n.get('node_id')
            print(f"   └── [{rel_type}] ──> {target}")
    else:
        print("Graph store does not support neighbor inspection.")

inspect_node("project_apollo")

---

## 6. 混合检索(GraphRAG)

精彩的部分来了。我们提出一个需要把线索串联起来才能回答的问题。

**查询**:*"谁负责那个 Python Web 框架项目?"*

**逻辑流程:**
1.  **向量搜索**:找到 "Project Apollo"(被描述为 Web 框架)。
2.  **图扩展**:在图中查看 "Project Apollo"。
3.  **发现**:看到 `(Alice)-[MANAGES]->(Project Apollo)`。
4.  **结果**:返回 Alice,尽管她的名字并未出现在项目描述文本中!

In [ ]:
query = "Who is responsible for the Python web framework project?"
print(f"Asking: '{query}'...\n")

results = context.retrieve(
    query,
    max_results=3,
    use_graph=True,         # Vital for finding Alice
    expand_graph=True,      # Hop to neighbors
    include_entities=True   # Return structured entity data
)

print(f"Retrieved {len(results)} context items:\n")

for i, res in enumerate(results, 1):
    print(f"{i}. [Score: {res['score']:.2f}] {res['content'][:120]}...")
    
    # Did we find graph connections?
    if 'related_entities' in res and res['related_entities']:
        print("   Graph Insights:")
        for ent in res['related_entities'][:3]:
            print(f"      - {ent.get('text', 'Entity')} ({ent.get('type', 'Unknown')})")
    print("")

---

## 7. 生命周期管理

生产系统需要维护。你可以查询历史、检查健康状况并清理旧数据。

### 对话历史

In [ ]:
# Get recent chat history for context window
history = context.conversation(
    conversation_id=session_id,
    limit=5
)

print(f"Chat History for {session_id}:")
for msg in history:
    print(f" - {msg['content']}")

### 系统健康与统计

In [ ]:
stats = context.stats()
print("System Vital Signs:")
print(f"   - Total Memories: {stats.get('total_items', 0)}")
print(f"   - Graph Nodes:    {stats.get('graph_stats', {}).get('node_count', 'N/A')}")
print(f"   - Graph Edges:    {stats.get('graph_stats', {}).get('edge_count', 'N/A')}")

## 总结

你已经使用 Semantica 的生产级模块成功构建了一个**具备上下文感知的智能体**。

**关键成果:**
1.  **持久化**:换用 FAISS 和 Neo4j 作为真实世界的存储。
2.  **GraphRAG**:演示了图关系如何提升检索准确率。
3.  **实体注入**:手动教会智能体了解业务关系。

该架构已可扩展到数百万向量和图节点。